# Notebook 5: Analyze Your Own Audio Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/williamedwardhahn/birdsong/blob/main/Notebook_5_Your_Own_Data.ipynb)

In this notebook you will:

1. **Upload** your own audio recordings to Google Drive
2. **Load** them into the notebook (supports `.m4a`, `.wav`, `.mp3`, and more)
3. **Visualize** each recording as a waveform and spectrogram
4. **Extract MFCC features** and **cluster** your recordings using an autoencoder

This is your chance to work with real data you collected yourself!

### Before You Start

**Important — use the same Google account everywhere.** This notebook connects to Google Drive to read your audio files. For that to work, the Google account you use to open this notebook in Colab must be the **same** Google account you use on your phone's Google Drive app. If you record audio on your phone and upload it to Drive with one account, but open Colab with a different account, the notebook won't be able to see your files. Check which account you're signed into on both devices before you start.

1. Go to **File → Save a copy in Drive**
2. Rename it with your name (e.g. `FirstName_LastName_Notebook_5.ipynb`)
3. Work in **your copy** from now on

**When you're done:** Click the **Share** button (top right), copy the link, and submit it to your instructor. Make sure sharing is set to **"Anyone with the link can view."**

**Colab tip:** Close any other notebooks you have open — Colab only lets you run one or two at a time. If it disconnects, use **Runtime → Disconnect and delete runtime**, then reopen.

## Before You Start: Record and Upload Your Audio Files

You need audio files in your Google Drive before running this notebook. Below are complete instructions for recording on your phone, renaming the files, and uploading them.

---

### Recording on iPhone (iOS)

iPhones have a built-in app called **Voice Memos** — no download needed.

**To record:**
1. Open the **Voice Memos** app (it looks like a black waveform icon — check the Utilities folder if you can't find it)
2. Tap the **red circle** button to start recording
3. Hold the phone toward the sound source (a bird, a person speaking, etc.)
4. Tap the **red square** button to stop recording
5. The recording appears in the list with a default name like "New Recording 1"

**To rename:**
1. Tap the recording name in the list
2. Tap the name again at the top — it becomes editable
3. Type a descriptive name like `cardinal_1` or `sparrow_morning` (use underscores instead of spaces)
4. Tap anywhere else to save

**To upload to Google Drive:**
1. Open the **Google Drive** app (download it from the App Store if needed — it's free)
2. Navigate to the **Audio** folder (the one you created in My Drive)
3. Tap the **+** button (bottom right)
4. Tap **Upload**
5. Tap **Browse** and navigate to **Voice Memos** — the recordings will be listed there
6. Select the file and tap **Upload**

**Alternative method:** In the Voice Memos app, tap the **three dots (...)** next to a recording, tap **Share**, then tap **Google Drive** from the share sheet. Choose the Audio folder and tap **Upload**.

---

### Recording on Android

Most Android phones come with a built-in recorder. The app name varies by manufacturer:

- **Samsung:** Voice Recorder (in the Samsung folder)
- **Google Pixel:** Recorder
- **Other brands:** look for "Recorder", "Voice Recorder", or "Sound Recorder" in the app drawer

**To record:**
1. Open the recorder app
2. Tap the **red circle** button to start recording
3. Hold the phone toward the sound source
4. Tap the **stop** button when done
5. Name the recording something descriptive like `cardinal_1` or `sparrow_morning`
6. Tap **Save**

**If the recorder saves as `.m4a` instead of `.wav`:** That's fine — this notebook supports `.m4a`, `.mp3`, `.wav`, `.ogg`, and `.flac`. No need to convert anything.

**To upload to Google Drive:**
1. Open the **Google Drive** app (it comes pre-installed on most Android phones)
2. Navigate to the **Audio** folder
3. Tap the **+** button (bottom right)
4. Tap **Upload**
5. Navigate to the recordings — they're usually in **Internal storage → Recordings** or **Internal storage → Voice Recorder**
6. Select the file(s) and tap **Done** or **Upload**

**Alternative method:** Open the phone's **Files** app, find the recording, tap **Share**, then tap **Google Drive**. Choose the Audio folder and tap **Save**.

---

### Tips for Good Recordings

- **Get close to the source** — the closer, the cleaner the recording
- **Minimize background noise** — wind, traffic, and other people talking will show up in the spectrogram
- **Record for at least 5 seconds** — very short clips don't have enough data for good analysis
- **Use descriptive names** — `robin_backyard_morning` is much better than `New Recording 47`
- **Record multiple examples** — 3–5 recordings per category gives the clustering more to work with

## Step 1: Setup & Mount Google Drive

This cell installs dependencies and connects to your Google Drive so we can access your files.

In [ ]:
# Install dependencies
!pip install -q pydub

# Import libraries
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from pydub import AudioSegment
from IPython.display import Audio, display

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Setup complete!")

## Step 2: Helper Functions

In [ ]:
def load_audio_file(file_path):
    """
    Load an audio file of any format using pydub, then convert to a numpy array.
    Returns:
        samples: float32 numpy array scaled to [-1, 1]
        sr: sample rate
    """
    audio = AudioSegment.from_file(file_path)
    samples = np.array(audio.get_array_of_samples())
    if audio.channels == 2:
        samples = samples.reshape((-1, 2)).mean(axis=1)
    samples = samples.astype(np.float32)
    max_val = float(2 ** (8 * audio.sample_width - 1))
    samples /= max_val
    return samples, audio.frame_rate


def plot_waveform(y, sr, title="Waveform"):
    """Plot the waveform of an audio signal."""
    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y, sr=sr)
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.show()


def plot_spectrogram(y, sr, title="Spectrogram"):
    """Plot the spectrogram of an audio signal."""
    S = np.abs(librosa.stft(y))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.show()


class Autoencoder(nn.Module):
    """Compresses input features down to 2 dimensions, then reconstructs them."""
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded


print("Helper functions ready.")

## Step 3: Find Your Audio Files

This cell looks in your `My Drive/Audio` folder and lists what it finds.

### How to set up your Audio folder:

1. Open a new tab and go to [drive.google.com](https://drive.google.com)
2. Make sure you're in **My Drive** (the top level — not inside any other folder)
3. Click **New → New folder** and name it exactly **`Audio`** (capital A, no spaces before or after)
4. Open the `Audio` folder
5. Drag and drop your audio files into it, or click **New → File upload**
   - Supported formats: `.m4a`, `.wav`, `.mp3`, `.ogg`, `.flac`
   - You can record audio on your phone and upload those files directly
   - Aim for **5 or more files** to get interesting clustering results

Your Drive should look like this:
```
My Drive/
  Audio/
    recording1.m4a
    recording2.mp3
    recording3.wav
    ...
```

### If no files are found when you run the cell below:

- **Wrong Google account?** The account you used to open Colab must be the same account where your Audio folder lives. Check the profile icon in the top-right corner of both Colab and Drive — the email address should match.
- **Wrong folder name?** It must be exactly `Audio` — not `audio`, not `My Audio`, not `Audio Files`
- **Folder in the wrong place?** It needs to be at the top level of My Drive, not inside another folder
- **Drive not mounted?** Go back and re-run the Setup cell (Step 1) — you should see a prompt asking you to allow access
- **Just uploaded?** Sometimes it takes a minute for new files to appear. Try running the cell again
- **Wrong file format?** Only `.m4a`, `.wav`, `.mp3`, `.ogg`, and `.flac` files are detected

In [ ]:
# Change this path if your files are in a different folder
AUDIO_FOLDER = '/content/drive/MyDrive/Audio'

# Find all audio files
AUDIO_EXTENSIONS = ('*.m4a', '*.wav', '*.mp3', '*.ogg', '*.flac')
audio_files = []
for ext in AUDIO_EXTENSIONS:
    audio_files.extend(glob.glob(os.path.join(AUDIO_FOLDER, ext)))
audio_files = sorted(audio_files)

print(f"Found {len(audio_files)} audio files in {AUDIO_FOLDER}:\n")
for i, f in enumerate(audio_files, 1):
    print(f"  {i}. {os.path.basename(f)}")

## Step 4: Visualize & Listen to Each File

For each audio file, we'll display the waveform, spectrogram, and an audio player.

In [ ]:
for file_path in audio_files:
    name = os.path.basename(file_path)
    print(f"\n{'='*60}")
    print(f"File: {name}")
    print(f"{'='*60}")

    try:
        y, sr = load_audio_file(file_path)
        print(f"  Sample rate: {sr} Hz  |  Duration: {len(y)/sr:.2f}s")

        plot_waveform(y, sr, title=f"{name} \u2014 Waveform")
        plot_spectrogram(y, sr, title=f"{name} \u2014 Spectrogram")

        print("  Click play to listen:")
        display(Audio(y, rate=sr))
    except Exception as e:
        print(f"  Error processing {name}: {e}")

## Step 5: Extract MFCC Features

MFCCs (Mel Frequency Cepstral Coefficients) capture the "shape" of each sound — what makes one recording sound different from another.

We extract 13 MFCCs from each file and summarize them as a mean and standard deviation, giving us 26 features per recording.

In [ ]:
N_MFCC = 13
features = []
file_names = []

for file_path in audio_files:
    try:
        y, sr = load_audio_file(file_path)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        feature_vec = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
        features.append(feature_vec)
        file_names.append(os.path.basename(file_path))
    except Exception as e:
        print(f"Error extracting features from {os.path.basename(file_path)}: {e}")

X = np.array(features)
print(f"Extracted features from {len(features)} files.")
print(f"Feature matrix shape: {X.shape}  (files x features)")

## Step 6: Train the Autoencoder & Cluster

We train an autoencoder to compress the 26 MFCC features down to just **2 dimensions** so we can plot them on a scatter plot.

Recordings that sound similar should appear close together on the plot.

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

# Train the autoencoder
model = Autoencoder(input_dim=X_scaled.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

NUM_EPOCHS = 2000
model.train()
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    encoded, decoded = model(X_tensor)
    loss = criterion(decoded, X_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}  \u2014  Loss: {loss.item():.4f}")

print("Training complete!")

In [ ]:
# Generate 2D embeddings
model.eval()
with torch.no_grad():
    embeddings, _ = model(X_tensor)
embeddings = embeddings.numpy()

# Plot each recording as a labeled point
plt.figure(figsize=(10, 7))
plt.scatter(embeddings[:, 0], embeddings[:, 1], s=80, alpha=0.7)

# Label each point with its filename
for i, name in enumerate(file_names):
    plt.annotate(name, (embeddings[i, 0], embeddings[i, 1]),
                 fontsize=8, ha='left', va='bottom')

plt.title("2D Embedding of Your Audio Files (Autoencoder on MFCC Features)", fontsize=13)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Recordings that sound similar should appear close together!")
print("Try adding more files to your Audio folder and re-running to see how the clusters change.")

## Bonus: Color by Group

The scatter plot above shows all your recordings as the same color. If you want to see whether similar recordings cluster together, you can **color them by group**.

The cell below automatically assigns colors based on alphabetical order — every 5 files get the same color. This works well if you name your files with a shared prefix for each category:

```
cardinal_1.mp3      ← Group A (red)
cardinal_2.mp3      ← Group A (red)
cardinal_3.mp3      ← Group A (red)
cardinal_4.mp3      ← Group A (red)
cardinal_5.mp3      ← Group A (red)
sparrow_1.mp3       ← Group B (blue)
sparrow_2.mp3       ← Group B (blue)
...
```

**To adjust:** Change `GROUP_SIZE` below to match how many files you have per category. For example, if you uploaded 3 recordings of each bird, set `GROUP_SIZE = 3`.

In [ ]:
GROUP_SIZE = 5  # Change this to match your data

# Assign group labels
group_labels = [chr(ord('A') + i // GROUP_SIZE) for i in range(len(file_names))]

plt.figure(figsize=(10, 7))
for label in sorted(set(group_labels)):
    mask = [g == label for g in group_labels]
    pts = embeddings[mask]
    plt.scatter(pts[:, 0], pts[:, 1], label=f"Group {label}", s=80, alpha=0.7)

plt.legend(fontsize=11)
plt.title("2D Embeddings Colored by Group", fontsize=13)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## What You Learned

| What you did | Why it matters |
|-------------|----------------|
| **Uploaded** your own audio to Google Drive | You can analyze any audio you collect in the field or record on your phone |
| **Loaded** multiple audio formats | `pydub` handles .m4a, .mp3, .wav, .ogg, .flac — no manual conversion needed |
| **Visualized** your recordings | Waveforms and spectrograms reveal structure you can't hear |
| **Extracted MFCCs** from your data | Same feature extraction pipeline works on any audio |
| **Clustered** with an autoencoder | Similar-sounding recordings group together in 2D space |

**Experiment ideas:**
- Record different people speaking and see if the autoencoder separates them
- Record the same bird at different times and see how consistent the songs are
- Mix bird recordings with non-bird sounds — can the autoencoder tell them apart?

**You've completed all 6 notebooks!** You now have the skills to load, visualize, and analyze any audio data with Python.